In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, warnings
REPO = '/content/emg-thesis'
if os.path.exists(REPO):
    !git -C {REPO} pull
else:
    !git clone https://github.com/mshahnaza/emg-cross-session-mcu.git {REPO}
!pip install -q -r {REPO}/requirements.txt
sys.path.append(REPO + '/src')

from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

FEATURES_DIR = '/content/drive/MyDrive/ninapro/'
RESULTS_DIR = '/content/drive/MyDrive/ninapro/results/all/'
os.makedirs(RESULTS_DIR, exist_ok=True)
SUBJECTS = range(1, 11)

In [ ]:
from emg.data import load_features
from emg.summary import run_subject

for s in SUBJECTS:
    path = RESULTS_DIR + f'board_S{s}.csv'
    if os.path.exists(path):
        print(f'S{s}: done earlier')
        continue
    data, extra = load_features(FEATURES_DIR + f'features_S{s}.npz', s)
    rq1_table, renorm_df, board_df = run_subject(data, s)
    rq1_table.to_csv(RESULTS_DIR + f'rq1_S{s}.csv', index=False)
    renorm_df.to_csv(RESULTS_DIR + f'renorm_S{s}.csv', index=False)
    board_df.to_csv(path, index=False)
    print(f'S{s}: finished')

In [ ]:
import pandas as pd
from emg.summary import analyse

rq1_all = pd.concat([pd.read_csv(RESULTS_DIR + f'rq1_S{s}.csv') for s in SUBJECTS])
renorm_all = pd.concat([pd.read_csv(RESULTS_DIR + f'renorm_S{s}.csv') for s in SUBJECTS])
board_all = pd.concat([pd.read_csv(RESULTS_DIR + f'board_S{s}.csv') for s in SUBJECTS])

res = analyse(rq1_all, renorm_all, board_all)
for name, table in res.items():
    table.to_csv(RESULTS_DIR + f'all_{name}.csv')
    print(name)
    display(table)